# Audio Dataset Preparation (Versioned)

This notebook converts session-level audio recordings into window-level training and validation examples.

**NEW**: Creates versioned datasets with unique folder names based on parameters (window length, hop length, date).

**Input**: Session WAV files in `sessions/` directory

**Output**: 
- Windowed WAV files in `datasets/dataset_w{X}s_h{Y}s_{DATE}/train/` and `.../val/`
- Metadata CSV file with labels and split information
- Each parameter combination creates a separate, versioned dataset folder

**Process**:
1. Load session metadata from filenames
2. Perform session-level train/val split
3. Slice sessions into overlapping windows
4. Assign labels based on session label
5. Export windowed audio and metadata

## 1. Imports & Configuration

In [1]:
# Imports
import numpy as np
import librosa
import soundfile as sf
import pandas as pd
from pathlib import Path
import os
from datetime import datetime

In [2]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# Audio parameters
SAMPLE_RATE = 16000      # Hz
WINDOW_LENGTH = 1.5      # seconds - CHANGE THIS to create different datasets
HOP_LENGTH = 0.25        # seconds - CHANGE THIS to create different datasets

# Generate unique dataset directory name based on parameters
from datetime import datetime
timestamp = datetime.now().strftime('%Y%m%d')
dataset_name = f'dataset_w{WINDOW_LENGTH}s_h{HOP_LENGTH}s_{timestamp}'

# Paths
SESSIONS_DIR = Path('sessions')
DATASET_DIR = Path('datasets') / dataset_name  # Changed to 'datasets' parent folder
TRAIN_DIR = DATASET_DIR / 'train'
VAL_DIR = DATASET_DIR / 'val'
METADATA_FILE = DATASET_DIR / 'dataset_metadata.csv'

# Derived parameters
WINDOW_SAMPLES = int(WINDOW_LENGTH * SAMPLE_RATE)
HOP_SAMPLES = int(HOP_LENGTH * SAMPLE_RATE)

print("Configuration:")
print(f"  Sample rate: {SAMPLE_RATE} Hz")
print(f"  Window length: {WINDOW_LENGTH}s ({WINDOW_SAMPLES} samples)")
print(f"  Hop length: {HOP_LENGTH}s ({HOP_SAMPLES} samples)")
print(f"  Sessions directory: {SESSIONS_DIR}")
print(f"  Dataset name: {dataset_name}")
print(f"  Output directory: {DATASET_DIR}")

Configuration:
  Sample rate: 16000 Hz
  Window length: 1.5s (24000 samples)
  Hop length: 0.25s (4000 samples)
  Sessions directory: sessions
  Dataset name: dataset_w1.5s_h0.25s_20260314
  Output directory: datasets/dataset_w1.5s_h0.25s_20260314


## 2. Load Session Metadata

Parse filenames to extract session information:
- `session_id`: timestamp
- `session_label`: glucose_alarm or no_glucose_alarm
- `context`: everything after the second `__`

In [3]:
def parse_session_filename(filepath):
    """
    Parse session filename to extract metadata.
    
    Expected format: session_<timestamp>__<label>__<context>.wav
    
    Parameters:
    -----------
    filepath : Path
        Path to the session WAV file
    
    Returns:
    --------
    dict
        Dictionary with session_id, session_label, context, filepath
    """
    filename = filepath.stem  # Remove .wav extension
    
    # Split by '__' to get parts
    parts = filename.split('__')
    
    if len(parts) < 3:
        raise ValueError(f"Invalid filename format: {filename}")
    
    # Extract session_id (remove 'session_' prefix)
    session_id = parts[0].replace('session_', '')
    
    # Extract label
    session_label = parts[1]
    
    # Extract context (everything after second __)
    context = '__'.join(parts[2:])
    
    return {
        'session_id': session_id,
        'session_label': session_label,
        'context': context,
        'filepath': str(filepath)
    }

In [4]:
# Scan sessions directory
session_files = sorted(SESSIONS_DIR.glob('*.wav'))

print(f"Found {len(session_files)} session files\n")

# Parse metadata
sessions_data = []
for filepath in session_files:
    try:
        metadata = parse_session_filename(filepath)
        sessions_data.append(metadata)
        print(f"✓ {filepath.name}")
        print(f"  ID: {metadata['session_id']}, Label: {metadata['session_label']}, Context: {metadata['context']}")
    except Exception as e:
        print(f"✗ {filepath.name}: {e}")

# Create DataFrame
sessions_df = pd.DataFrame(sessions_data)

print(f"\nLoaded {len(sessions_df)} sessions")
print(f"\nSession label distribution:")
print(sessions_df['session_label'].value_counts())

Found 4 session files

✓ session_20260111_204751__glucose_alarm__no_background_noise.wav
  ID: 20260111_204751, Label: glucose_alarm, Context: no_background_noise
✓ session_20260111_205217__glucose_alarm__background_noise.wav
  ID: 20260111_205217, Label: glucose_alarm, Context: background_noise
✓ session_20260111_205814__no_glucose_alarm__alarms.wav
  ID: 20260111_205814, Label: no_glucose_alarm, Context: alarms
✓ session_20260111_210208__no_glucose_alarm__background_noise.wav
  ID: 20260111_210208, Label: no_glucose_alarm, Context: background_noise

Loaded 4 sessions

Session label distribution:
session_label
glucose_alarm       2
no_glucose_alarm    2
Name: count, dtype: int64


In [5]:
# Display session metadata
sessions_df

,session_id,session_label,context,filepath
0,20260111_204751,glucose_alarm,no_background_noise,sessions/session_20260111_204751__glucose_alar...
1,20260111_205217,glucose_alarm,background_noise,sessions/session_20260111_205217__glucose_alar...
2,20260111_205814,no_glucose_alarm,alarms,sessions/session_20260111_205814__no_glucose_a...
3,20260111_210208,no_glucose_alarm,background_noise,sessions/session_20260111_210208__no_glucose_a...


## 3. Train / Validation Split

**Important**: Split at the session level, NOT at the window level.

This ensures that windows from the same session don't appear in both train and validation sets.

**Hardcoded split**:
- 1 glucose_alarm session → train
- 1 glucose_alarm session → val
- 1 no_glucose_alarm session → train
- 1 no_glucose_alarm session → val

In [6]:
# Separate sessions by label
glucose_alarm_sessions = sessions_df[sessions_df['session_label'] == 'glucose_alarm'].copy()
no_glucose_alarm_sessions = sessions_df[sessions_df['session_label'] == 'no_glucose_alarm'].copy()

print(f"Glucose alarm sessions: {len(glucose_alarm_sessions)}")
print(f"No glucose alarm sessions: {len(no_glucose_alarm_sessions)}")

# Hardcoded split assignment
# Assign first session of each type to train, second to val
sessions_df['split'] = None

# Glucose alarm split
if len(glucose_alarm_sessions) >= 2:
    glucose_train_idx = glucose_alarm_sessions.index[0]
    glucose_val_idx = glucose_alarm_sessions.index[1]
    sessions_df.loc[glucose_train_idx, 'split'] = 'train'
    sessions_df.loc[glucose_val_idx, 'split'] = 'val'
    print(f"\nGlucose alarm:")
    print(f"  Train: {sessions_df.loc[glucose_train_idx, 'session_id']}")
    print(f"  Val:   {sessions_df.loc[glucose_val_idx, 'session_id']}")
elif len(glucose_alarm_sessions) == 1:
    glucose_train_idx = glucose_alarm_sessions.index[0]
    sessions_df.loc[glucose_train_idx, 'split'] = 'train'
    print(f"\n⚠️  Warning: Only 1 glucose_alarm session found, assigning to train")
else:
    print(f"\n⚠️  Warning: No glucose_alarm sessions found")

# No glucose alarm split
if len(no_glucose_alarm_sessions) >= 2:
    no_glucose_train_idx = no_glucose_alarm_sessions.index[0]
    no_glucose_val_idx = no_glucose_alarm_sessions.index[1]
    sessions_df.loc[no_glucose_train_idx, 'split'] = 'train'
    sessions_df.loc[no_glucose_val_idx, 'split'] = 'val'
    print(f"\nNo glucose alarm:")
    print(f"  Train: {sessions_df.loc[no_glucose_train_idx, 'session_id']}")
    print(f"  Val:   {sessions_df.loc[no_glucose_val_idx, 'session_id']}")
elif len(no_glucose_alarm_sessions) == 1:
    no_glucose_train_idx = no_glucose_alarm_sessions.index[0]
    sessions_df.loc[no_glucose_train_idx, 'split'] = 'train'
    print(f"\n⚠️  Warning: Only 1 no_glucose_alarm session found, assigning to train")
else:
    print(f"\n⚠️  Warning: No no_glucose_alarm sessions found")

# Mark any remaining sessions as unused
sessions_df['split'] = sessions_df['split'].fillna('unused')

print(f"\nSplit summary:")
print(sessions_df.groupby(['split', 'session_label']).size())

Glucose alarm sessions: 2
No glucose alarm sessions: 2

Glucose alarm:
  Train: 20260111_204751
  Val:   20260111_205217

No glucose alarm:
  Train: 20260111_205814
  Val:   20260111_210208

Split summary:
split  session_label   
train  glucose_alarm       1
       no_glucose_alarm    1
val    glucose_alarm       1
       no_glucose_alarm    1
dtype: int64


In [7]:
# Display sessions with split assignment
sessions_df[['session_id', 'session_label', 'context', 'split']]

,session_id,session_label,context,split
0,20260111_204751,glucose_alarm,no_background_noise,train
1,20260111_205217,glucose_alarm,background_noise,val
2,20260111_205814,no_glucose_alarm,alarms,train
3,20260111_210208,no_glucose_alarm,background_noise,val


## 4. Slice Sessions into Overlapping Windows

For each session:
1. Load audio (resample to 16kHz, convert to mono)
2. Slice into overlapping windows
3. Track window metadata (session_id, window_index, start_time)

In [8]:
def slice_audio_into_windows(audio, sample_rate, window_samples, hop_samples):
    """
    Slice audio into overlapping windows.
    
    Parameters:
    -----------
    audio : np.ndarray
        Audio signal (mono)
    sample_rate : int
        Sample rate in Hz
    window_samples : int
        Window length in samples
    hop_samples : int
        Hop length in samples
    
    Returns:
    --------
    list of dict
        List of windows with metadata
    """
    windows = []
    
    # Calculate number of windows
    num_samples = len(audio)
    
    window_idx = 0
    start_sample = 0
    
    while start_sample + window_samples <= num_samples:
        # Extract window
        window_audio = audio[start_sample:start_sample + window_samples]
        
        # Calculate start time in seconds
        start_time = start_sample / sample_rate
        
        windows.append({
            'window_index': window_idx,
            'start_time_seconds': start_time,
            'audio': window_audio
        })
        
        window_idx += 1
        start_sample += hop_samples
    
    return windows

In [9]:
# Process all sessions and create windows
all_windows = []

print("Processing sessions...\n")

for idx, row in sessions_df.iterrows():
    session_id = row['session_id']
    session_label = row['session_label']
    context = row['context']
    split = row['split']
    filepath = row['filepath']
    
    # Skip unused sessions
    if split == 'unused':
        print(f"⊘ Skipping {session_id} (unused)")
        continue
    
    print(f"Processing {session_id} ({session_label}, {split})...")
    
    # Load audio
    audio, sr = librosa.load(filepath, sr=SAMPLE_RATE, mono=True)
    
    duration = len(audio) / sr
    print(f"  Loaded: {duration:.2f}s, {len(audio)} samples")
    
    # Slice into windows
    windows = slice_audio_into_windows(audio, SAMPLE_RATE, WINDOW_SAMPLES, HOP_SAMPLES)
    
    print(f"  Created {len(windows)} windows")
    
    # Add session metadata to each window
    for window in windows:
        all_windows.append({
            'session_id': session_id,
            'session_label': session_label,
            'context': context,
            'split': split,
            'window_index': window['window_index'],
            'start_time_seconds': window['start_time_seconds'],
            'audio': window['audio']
        })

print(f"\nTotal windows created: {len(all_windows)}")

Processing sessions...

Processing 20260111_204751 (glucose_alarm, train)...
  Loaded: 180.00s, 2880000 samples
  Created 715 windows
Processing 20260111_205217 (glucose_alarm, val)...
  Loaded: 180.00s, 2880000 samples
  Created 715 windows
Processing 20260111_205814 (no_glucose_alarm, train)...
  Loaded: 180.00s, 2880000 samples
  Created 715 windows
Processing 20260111_210208 (no_glucose_alarm, val)...
  Loaded: 180.00s, 2880000 samples
  Created 715 windows

Total windows created: 2860


## 5. Window-Level Labels

Assign binary labels based on session label:
- `glucose_alarm` → label = 1
- `no_glucose_alarm` → label = 0

In [10]:
# Assign labels
for window in all_windows:
    if window['session_label'] == 'glucose_alarm':
        window['label'] = 1
    elif window['session_label'] == 'no_glucose_alarm':
        window['label'] = 0
    else:
        raise ValueError(f"Unknown session label: {window['session_label']}")

# Create DataFrame for analysis
windows_df = pd.DataFrame([{k: v for k, v in w.items() if k != 'audio'} for w in all_windows])

print("Window-level dataset summary:\n")
print(f"Total windows: {len(windows_df)}")
print(f"\nWindows per split:")
print(windows_df['split'].value_counts())
print(f"\nClass balance per split:")
print(windows_df.groupby(['split', 'label']).size())

Window-level dataset summary:

Total windows: 2860

Windows per split:
split
train    1430
val      1430
Name: count, dtype: int64

Class balance per split:
split  label
train  0        715
       1        715
val    0        715
       1        715
dtype: int64


In [11]:
# Display sample of windows
windows_df.head(10)

,session_id,session_label,context,split,window_index,start_time_seconds,label
0,20260111_204751,glucose_alarm,no_background_noise,train,0,0.00,1
1,20260111_204751,glucose_alarm,no_background_noise,train,1,0.25,1
2,20260111_204751,glucose_alarm,no_background_noise,train,2,0.50,1
3,20260111_204751,glucose_alarm,no_background_noise,train,3,0.75,1
4,20260111_204751,glucose_alarm,no_background_noise,train,4,1.00,1
5,20260111_204751,glucose_alarm,no_background_noise,train,5,1.25,1
6,20260111_204751,glucose_alarm,no_background_noise,train,6,1.50,1
7,20260111_204751,glucose_alarm,no_background_noise,train,7,1.75,1
8,20260111_204751,glucose_alarm,no_background_noise,train,8,2.00,1
9,20260111_204751,glucose_alarm,no_background_noise,train,9,2.25,1


## 6. Export Dataset

Save windowed audio files and metadata CSV to the versioned dataset folder.

In [12]:
# Create output directories
DATASET_DIR.mkdir(parents=True, exist_ok=True)  # parents=True creates 'datasets' folder too
TRAIN_DIR.mkdir(exist_ok=True)
VAL_DIR.mkdir(exist_ok=True)

print(f"Created directories:")
print(f"  {TRAIN_DIR}")
print(f"  {VAL_DIR}")

Created directories:
  datasets/dataset_w1.5s_h0.25s_20260314/train
  datasets/dataset_w1.5s_h0.25s_20260314/val


In [13]:
windows_df

,session_id,session_label,context,split,window_index,start_time_seconds,label
0,20260111_204751,glucose_alarm,no_background_noise,train,0,0.00,1
1,20260111_204751,glucose_alarm,no_background_noise,train,1,0.25,1
2,20260111_204751,glucose_alarm,no_background_noise,train,2,0.50,1
3,20260111_204751,glucose_alarm,no_background_noise,train,3,0.75,1
4,20260111_204751,glucose_alarm,no_background_noise,train,4,1.00,1
...,...,...,...,...,...,...,...
2855,20260111_210208,no_glucose_alarm,background_noise,val,710,177.50,0
2856,20260111_210208,no_glucose_alarm,background_noise,val,711,177.75,0
2857,20260111_210208,no_glucose_alarm,background_noise,val,712,178.00,0
2858,20260111_210208,no_glucose_alarm,background_noise,val,713,178.25,0


In [14]:
# Save windowed audio files
metadata_records = []

print("\nSaving windowed audio files...\n")

for window in all_windows:
    session_id = window['session_id']
    window_index = window['window_index']
    split = window['split']
    audio = window['audio']
    
    # Generate filename
    filename = f"{session_id}_window_{window_index:04d}.wav"
    
    # Determine output directory
    if split == 'train':
        output_path = TRAIN_DIR / filename
    elif split == 'val':
        output_path = VAL_DIR / filename
    else:
        continue  # Skip unused
    
    # Save audio
    sf.write(output_path, audio, SAMPLE_RATE)
    
    # Record metadata
    metadata_records.append({
        'filename': filename,
        'session_id': session_id,
        'window_index': window_index,
        'label': window['label'],
        'split': split,
        'start_time_seconds': window['start_time_seconds'],
        'context': window['context']
    })

print(f"Saved {len(metadata_records)} window files")


Saving windowed audio files...

Saved 2860 window files


In [15]:
# Create and save metadata CSV
metadata_df = pd.DataFrame(metadata_records)
metadata_df.to_csv(METADATA_FILE, index=False)

print(f"\nSaved metadata to: {METADATA_FILE}")
print(f"\nMetadata columns: {list(metadata_df.columns)}")
print(f"Total records: {len(metadata_df)}")


Saved metadata to: datasets/dataset_w1.5s_h0.25s_20260314/dataset_metadata.csv

Metadata columns: ['filename', 'session_id', 'window_index', 'label', 'split', 'start_time_seconds', 'context']
Total records: 2860


## 7. Sanity Checks

Verify the versioned dataset was created correctly.

In [16]:
# Count files in each directory
train_files = list(TRAIN_DIR.glob('*.wav'))
val_files = list(VAL_DIR.glob('*.wav'))

print("File counts:")
print(f"  Train: {len(train_files)} files")
print(f"  Val:   {len(val_files)} files")
print(f"  Total: {len(train_files) + len(val_files)} files")

File counts:
  Train: 1430 files
  Val:   1430 files
  Total: 2860 files


In [17]:
# Class balance per split
print("\nClass balance:")
print("\nTrain set:")
train_balance = metadata_df[metadata_df['split'] == 'train']['label'].value_counts()
print(train_balance)
print(f"  Class 0: {train_balance.get(0, 0)} windows ({train_balance.get(0, 0) / len(metadata_df[metadata_df['split'] == 'train']) * 100:.1f}%)")
print(f"  Class 1: {train_balance.get(1, 0)} windows ({train_balance.get(1, 0) / len(metadata_df[metadata_df['split'] == 'train']) * 100:.1f}%)")

print("\nValidation set:")
val_balance = metadata_df[metadata_df['split'] == 'val']['label'].value_counts()
print(val_balance)
print(f"  Class 0: {val_balance.get(0, 0)} windows ({val_balance.get(0, 0) / len(metadata_df[metadata_df['split'] == 'val']) * 100:.1f}%)")
print(f"  Class 1: {val_balance.get(1, 0)} windows ({val_balance.get(1, 0) / len(metadata_df[metadata_df['split'] == 'val']) * 100:.1f}%)")


Class balance:

Train set:
label
1    715
0    715
Name: count, dtype: int64
  Class 0: 715 windows (50.0%)
  Class 1: 715 windows (50.0%)

Validation set:
label
1    715
0    715
Name: count, dtype: int64
  Class 0: 715 windows (50.0%)
  Class 1: 715 windows (50.0%)


In [18]:
# Verify no session leakage between train and val
train_sessions = set(metadata_df[metadata_df['split'] == 'train']['session_id'].unique())
val_sessions = set(metadata_df[metadata_df['split'] == 'val']['session_id'].unique())

overlap = train_sessions & val_sessions

print("\nSession leakage check:")
print(f"  Train sessions: {sorted(train_sessions)}")
print(f"  Val sessions:   {sorted(val_sessions)}")
print(f"  Overlap:        {overlap if overlap else 'None ✓'}")

if overlap:
    print("\n⚠️  WARNING: Session leakage detected!")
else:
    print("\n✓ No session leakage - train and val are properly separated")


Session leakage check:
  Train sessions: ['20260111_204751', '20260111_205814']
  Val sessions:   ['20260111_205217', '20260111_210208']
  Overlap:        None ✓

✓ No session leakage - train and val are properly separated


In [19]:
# Display sample metadata
print("\nSample metadata (first 10 windows):")
metadata_df.head(10)


Sample metadata (first 10 windows):


,filename,session_id,window_index,label,split,start_time_seconds,context
0,20260111_204751_window_0000.wav,20260111_204751,0,1,train,0.00,no_background_noise
1,20260111_204751_window_0001.wav,20260111_204751,1,1,train,0.25,no_background_noise
2,20260111_204751_window_0002.wav,20260111_204751,2,1,train,0.50,no_background_noise
3,20260111_204751_window_0003.wav,20260111_204751,3,1,train,0.75,no_background_noise
4,20260111_204751_window_0004.wav,20260111_204751,4,1,train,1.00,no_background_noise
5,20260111_204751_window_0005.wav,20260111_204751,5,1,train,1.25,no_background_noise
6,20260111_204751_window_0006.wav,20260111_204751,6,1,train,1.50,no_background_noise
7,20260111_204751_window_0007.wav,20260111_204751,7,1,train,1.75,no_background_noise
8,20260111_204751_window_0008.wav,20260111_204751,8,1,train,2.00,no_background_noise
9,20260111_204751_window_0009.wav,20260111_204751,9,1,train,2.25,no_background_noise


## Summary

Dataset preparation complete!

**Output structure:**
```
datasets/
└── dataset_w{WINDOW_LENGTH}s_h{HOP_LENGTH}s_{DATE}/
    ├── train/
    │   ├── <session_id>_window_0000.wav
    │   ├── <session_id>_window_0001.wav
    │   └── ...
    ├── val/
    │   ├── <session_id>_window_0000.wav
    │   ├── <session_id>_window_0001.wav
    │   └── ...
    └── dataset_metadata.csv
```

**Example:** `datasets/dataset_w1.5s_h0.25s_20260314/`

**To create another dataset with different parameters:**
1. Change `WINDOW_LENGTH` and/or `HOP_LENGTH` in the configuration cell
2. Run all cells again
3. A new versioned dataset folder will be created automatically

**Next steps:**
1. Load windowed audio for training
2. Compute spectrograms or other features
3. Train CNN model
4. Evaluate on validation set